# Lithography Ops AI — Stage D: Observability & Tracing (Colab)

Stages A–C built and evaluated the intelligence. Stage D makes every run
**observable**: a lightweight tracer records each step of a pipeline run as a
timed, nested **span**, so a finished run is a full, replayable record of what
happened — which agent ran, what it retrieved, and how long each step took.

This is the same idea as production tracing tools (OpenTelemetry-style spans),
but with **zero external dependencies** — it runs instantly, no install, no
model, no GPU. In the full project this tracer is built into the Coordinator and
saved to SQLite next to the audit trail; here we demonstrate it standalone.

Run the cells top to bottom (Shift+Enter).

## 1. The tracer

A span is a named, timed step. Spans nest, so a trace is a tree.

In [ ]:
import time, uuid, json
from contextlib import contextmanager
from dataclasses import dataclass, field

@dataclass
class Span:
    name: str; start_ms: float; end_ms: float = 0.0
    depth: int = 0; attributes: dict = field(default_factory=dict)
    @property
    def duration_ms(self):
        return round(self.end_ms - self.start_ms, 2)

class Tracer:
    def __init__(self, run_id=None):
        self.run_id = run_id or f'RUN-{uuid.uuid4().hex[:8]}'
        self.spans = []; self._depth = 0
    @contextmanager
    def span(self, name, **attrs):
        s = Span(name, time.perf_counter()*1000, depth=self._depth, attributes=attrs)
        self.spans.append(s); self._depth += 1
        try:
            yield s
        finally:
            self._depth -= 1; s.end_ms = time.perf_counter()*1000
    def total_ms(self):
        return round(sum(s.duration_ms for s in self.spans if s.depth==0), 2)
    def summary(self):
        out = [f'Trace {self.run_id}  (total {self.total_ms()} ms)']
        for s in self.spans:
            attr = ('  ' + ', '.join(f'{k}={v}' for k,v in s.attributes.items())) if s.attributes else ''
            out.append('  '*s.depth + f'\u2514 {s.name}: {s.duration_ms} ms{attr}')
        return '\n'.join(out)

print('Tracer ready.')

## 2. A mock pipeline run

We simulate a coordinator running its agents (with tiny sleeps standing in for real work) so you can watch the trace capture the structure and timing.

In [ ]:
def mock_run(machine_id, tracer):
    with tracer.span('coordinator', machine_id=machine_id):
        with tracer.span('agent:Monitoring'):
            time.sleep(0.030)   # ML health scoring is the heavy step
        with tracer.span('agent:IncidentTriage'):
            time.sleep(0.001)
        with tracer.span('agent:Knowledge', subsystem='cooling'):
            with tracer.span('tool:semantic_search', k=2):
                time.sleep(0.006)
            tracer.spans[-2].attributes['docs'] = 2
        with tracer.span('agent:Planning'):
            time.sleep(0.002)
        with tracer.span('agent:ShiftHandover'):
            time.sleep(0.001)

t = Tracer()
mock_run('LITHO-EUV-03', t)
print(t.summary())

## 3. Read the trace

The output above is a flight recorder for the run. Notice you can immediately see the **bottleneck** (Monitoring — the ML scoring) and confirm the Knowledge agent used semantic search and retrieved 2 documents. That is exactly the kind of question observability answers: *what happened, in what order, and how long?*

## 4. Visualise the trace as a timeline

In [ ]:
import matplotlib.pyplot as plt

labels = ['  '*s.depth + s.name for s in t.spans]
durs = [s.duration_ms for s in t.spans]
plt.figure(figsize=(8, 3.2))
plt.barh(range(len(labels)), durs, color='#39c5cf')
plt.yticks(range(len(labels)), labels)
plt.gca().invert_yaxis()
plt.xlabel('milliseconds')
plt.title(f'Run trace {t.run_id} — total {t.total_ms()} ms')
for i, d in enumerate(durs):
    plt.text(d, i, f' {d}', va='center', fontsize=8)
plt.tight_layout(); plt.show()

## 5. Compare two runs

Traces make it easy to compare runs — for example, a healthy machine vs a failing one, or (as here) two machines. Each run gets its own trace id.

In [ ]:
def mock_run_light(machine_id, tracer):
    with tracer.span('coordinator', machine_id=machine_id):
        with tracer.span('agent:Monitoring'):
            time.sleep(0.028)
        with tracer.span('agent:IncidentTriage'):
            time.sleep(0.001)
        with tracer.span('agent:Knowledge', subsystem='reticle stage'):
            with tracer.span('tool:semantic_search', k=2):
                time.sleep(0.005)
        with tracer.span('agent:Planning'):
            time.sleep(0.002)
        with tracer.span('agent:ShiftHandover'):
            time.sleep(0.001)

t2 = Tracer()
mock_run_light('LITHO-EUV-02', t2)
print(t.summary())
print()
print(t2.summary())

## 6. Save a trace (as JSON)

In the full project traces are written to SQLite next to the audit trail. Here we show the same data as JSON — queryable, auditable, replayable.

In [ ]:
trace_json = {
    'run_id': t.run_id, 'total_ms': t.total_ms(),
    'spans': [{'name': s.name, 'depth': s.depth,
               'duration_ms': s.duration_ms, 'attributes': s.attributes}
              for s in t.spans]
}
print(json.dumps(trace_json, indent=2))

## What you just built

A working **observability layer**: every pipeline run is captured as timed,
nested spans, so you can see which agent fired, what it retrieved, and where the
time went — then persist and replay it.

**Interview one-liner:** *"Every run is traced end to end with nested spans —
which agent fired, whether it used semantic or keyword retrieval, how many docs
it pulled, and the timing of each step — persisted next to the audit trail, the
same pattern production systems use with OpenTelemetry."*

This completes the four-stage pipeline: **A** semantic retrieval, **B** grounded
generation, **C** evaluation, and **D** observability.